# NSE Daily Stocks + NIFTY 50 — Full Fixed Incremental Downloader

This notebook is **filesystem-first for BOTH stocks and NIFTY 50**.

## Core rule

**Parquet filesystem state is authoritative.**

For both datasets:

- Existing Parquet → **never request/download**
- Missing Parquet → **queue for download/retry**
- Manifest status never overrides filesystem state
- Missing files are retried even if an old manifest says `downloaded`, `failed`, `error`, or `not_found`
- A complete preflight scan happens before network downloads
- A file appearing after queue creation causes an immediate abort rather than overwrite

## Storage

- Stocks: `/content/drive/MyDrive/quant/data/parquet`
- NIFTY 50: `/content/drive/MyDrive/quant/data/indices/nifty50`

## NIFTY-specific protection

The current NIFTY endpoint can return records outside the requested date range. The notebook:

1. logs the actual returned range,
2. logs out-of-range records,
3. filters locally,
4. hard-asserts that no out-of-range record survives,
5. saves only dates that were actually missing.

NIFTY API chunks with **zero missing dates are never requested**.


In [ ]:
# 1. Install and import dependencies

!pip -q install pyarrow requests pandas tqdm

import io
import json
import random
import time
import zipfile
from pathlib import Path
from datetime import date, timedelta, datetime

import pandas as pd
import requests
from tqdm.auto import tqdm


In [ ]:
# 2. Mount Google Drive

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. Configuration

BASE_DIR = Path("/content/drive/MyDrive/quant")

STOCK_DIR = BASE_DIR / "data" / "parquet"
NIFTY_DIR = BASE_DIR / "data" / "indices" / "nifty50"
MANIFEST_DIR = BASE_DIR / "data" / "manifests"

STOCK_MANIFEST = MANIFEST_DIR / "nse_stock_download_manifest.jsonl"
NIFTY_MANIFEST = MANIFEST_DIR / "nifty50_download_manifest.jsonl"

for p in [STOCK_DIR, NIFTY_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Historical range.
START_DATE = date(2011, 1, 1)
END_DATE = date.today()

# Retry/network settings.
MAX_RETRIES = 4
REQUEST_SLEEP_SECONDS = 0.20

# NIFTY API request chunk size.
# The endpoint has been observed to return a wider range than requested,
# so local filtering is mandatory.
NIFTY_CHUNK_DAYS = 150

DOWNLOAD_STOCKS = True
DOWNLOAD_NIFTY = True

print("Range :", START_DATE, "->", END_DATE)
print("Stock :", STOCK_DIR)
print("NIFTY :", NIFTY_DIR)


In [ ]:
# 4. Common helpers

STOCK_COLUMNS = [
    "date",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

def iso(d):
    return pd.Timestamp(d).date().isoformat()

def date_range_days(start, end):
    cur = pd.Timestamp(start).date()
    end = pd.Timestamp(end).date()

    while cur <= end:
        yield cur
        cur += timedelta(days=1)

def parquet_path(directory, day):
    return Path(directory) / f"{iso(day)}.parquet"

# Compatibility aliases for older notebook naming.
parquet_path_for_day = parquet_path

def append_jsonl(path, record):
    with Path(path).open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, default=str) + "\n")

def append_manifest(record):
    append_jsonl(STOCK_MANIFEST, record)

def load_latest_manifest(path):
    latest = {}
    path = Path(path)

    if not path.exists():
        return latest

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
                if row.get("date"):
                    latest[row["date"]] = row
            except Exception:
                # A malformed historical manifest entry must not block sync.
                continue

    return latest

def atomic_to_parquet(df, path):
    path = Path(path)
    tmp = path.with_suffix(".parquet.tmp")

    if tmp.exists():
        tmp.unlink()

    df.to_parquet(
        tmp,
        index=False,
        engine="pyarrow",
    )

    # Never overwrite.
    if path.exists():
        tmp.unlink()
        raise RuntimeError(
            f"Refusing to overwrite existing Parquet: {path}"
        )

    tmp.replace(path)

def validate_parquet(path, required_columns):
    try:
        df = pd.read_parquet(
            path,
            engine="pyarrow",
        )

        missing = set(required_columns) - set(df.columns)

        if missing:
            return False, f"missing columns: {sorted(missing)}"

        if df.empty:
            return False, "empty parquet"

        return True, f"{len(df):,} rows"

    except Exception as exc:
        return False, repr(exc)


In [ ]:
# 5. NSE session and Bhavcopy URL builders

LEGACY_CUTOFF = date(2024, 7, 5)

def build_nse_session():
    session = requests.Session()

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/140.0.0.0 Safari/537.36"
        ),
        "Accept": "*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    })

    try:
        response = session.get(
            "https://www.nseindia.com",
            timeout=30,
        )
        print("NSE session:", response.status_code)
    except Exception as exc:
        print("NSE session warning:", repr(exc))

    return session

nse_session = build_nse_session()

def nse_stock_url(day):
    day = pd.Timestamp(day).date()

    if day <= LEGACY_CUTOFF:

        month = day.strftime("%b").upper()

        filename = (
            f"cm{day.strftime('%d')}"
            f"{month}"
            f"{day.year}"
            f"bhav.csv.zip"
        )

        return (
            "https://nsearchives.nseindia.com/content/historical/"
            f"EQUITIES/{day.year}/{month}/{filename}"
        )

    filename = (
        "BhavCopy_NSE_CM_0_0_0_"
        f"{day.strftime('%Y%m%d')}_F_0000.csv.zip"
    )

    return (
        "https://nsearchives.nseindia.com/content/cm/"
        f"{filename}"
    )

print("Legacy:", nse_stock_url(date(2024, 7, 5)))
print("UDiFF :", nse_stock_url(date(2024, 7, 8)))


In [ ]:
# 6. Parse NSE Bhavcopy

def normalize_stock_bhavcopy(raw_bytes, day):

    with zipfile.ZipFile(
        io.BytesIO(raw_bytes)
    ) as archive:

        csv_files = [
            name
            for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]

        if not csv_files:
            raise RuntimeError(
                f"No CSV found inside ZIP: {archive.namelist()}"
            )

        csv_bytes = archive.read(csv_files[0])

    try:
        text = csv_bytes.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = csv_bytes.decode("latin-1")

    df = pd.read_csv(
        io.StringIO(text)
    )

    df.columns = [
        str(c)
        .strip()
        .upper()
        .replace(" ", "_")
        for c in df.columns
    ]

    def find_column(*names):
        for name in names:
            if name in df.columns:
                return name
        return None

    symbol_col = find_column("SYMBOL")
    series_col = find_column("SERIES")
    open_col = find_column("OPEN")
    high_col = find_column("HIGH")
    low_col = find_column("LOW")
    close_col = find_column("CLOSE")

    volume_col = find_column(
        "TOTTRDQTY",
        "TTL_TRD_QNTY",
        "TOTAL_TRADES_QUANTITY",
        "TOTAL_TRADE_QUANTITY",
    )

    mapping = {
        "SYMBOL": symbol_col,
        "OPEN": open_col,
        "HIGH": high_col,
        "LOW": low_col,
        "CLOSE": close_col,
        "VOLUME": volume_col,
    }

    missing = [
        name
        for name, value in mapping.items()
        if value is None
    ]

    if missing:
        raise RuntimeError(
            f"Could not map NSE columns {missing}. "
            f"Actual columns: {list(df.columns)}"
        )

    out = pd.DataFrame({
        "date": pd.Timestamp(day).normalize(),
        "symbol": df[symbol_col].astype(str).str.strip(),
        "open": pd.to_numeric(
            df[open_col],
            errors="coerce",
        ),
        "high": pd.to_numeric(
            df[high_col],
            errors="coerce",
        ),
        "low": pd.to_numeric(
            df[low_col],
            errors="coerce",
        ),
        "close": pd.to_numeric(
            df[close_col],
            errors="coerce",
        ),
        "volume": pd.to_numeric(
            df[volume_col],
            errors="coerce",
        ),
    })

    # Keep equity series where SERIES exists.
    if series_col:
        equity = (
            df[series_col]
            .astype(str)
            .str.strip()
            .str.upper()
            .eq("EQ")
        )

        out = out.loc[equity].copy()

    out = out.dropna(
        subset=[
            "symbol",
            "open",
            "high",
            "low",
            "close",
        ]
    )

    invalid_ohlc = (
        (out["high"] < out["low"])
        | (out["high"] < out["open"])
        | (out["high"] < out["close"])
        | (out["low"] > out["open"])
        | (out["low"] > out["close"])
        | (
            out[
                ["open", "high", "low", "close"]
            ] <= 0
        ).any(axis=1)
    )

    if invalid_ohlc.any():
        raise RuntimeError(
            f"{int(invalid_ohlc.sum())} invalid OHLC rows "
            f"detected for {day}"
        )

    return (
        out[STOCK_COLUMNS]
        .sort_values("symbol")
        .reset_index(drop=True)
    )


In [ ]:
# 7. Download one NSE stock date

def download_stock_day(day):

    day = pd.Timestamp(day).date()

    path = parquet_path(
        STOCK_DIR,
        day,
    )

    # NEVER request an existing file.
    if path.exists():
        raise RuntimeError(
            f"ABORT: stock Parquet already exists: {path}"
        )

    url = nse_stock_url(day)

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            response = nse_session.get(
                url,
                timeout=60,
            )

            if response.status_code == 404:

                return {
                    "status": "not_found",
                    "date": iso(day),
                    "http_status": 404,
                    "url": url,
                }

            response.raise_for_status()

            if len(response.content) < 100:
                raise RuntimeError(
                    f"Suspiciously small response: "
                    f"{len(response.content)} bytes"
                )

            df = normalize_stock_bhavcopy(
                response.content,
                day,
            )

            # Never overwrite if another process created it.
            if path.exists():
                raise RuntimeError(
                    f"ABORT: stock Parquet appeared before save: {path}"
                )

            atomic_to_parquet(
                df,
                path,
            )

            return {
                "status": "downloaded",
                "date": iso(day),
                "rows": len(df),
                "bytes": len(response.content),
                "http_status": response.status_code,
                "url": url,
            }

        except Exception as exc:

            last_error = repr(exc)

            if attempt < MAX_RETRIES:

                wait = min(
                    30,
                    2 ** (attempt - 1)
                    + random.random(),
                )

                time.sleep(wait)

    return {
        "status": "failed",
        "date": iso(day),
        "message": last_error,
        "url": url,
    }


In [ ]:
# 8. STOCK FILESYSTEM INVENTORY
#
# This is the authoritative source for deciding what to download.

candidates = list(
    date_range_days(
        START_DATE,
        END_DATE,
    )
)

stock_existing = []
stock_missing = []

for day in candidates:

    path = parquet_path(
        STOCK_DIR,
        day,
    )

    if path.exists():
        stock_existing.append(day)
    else:
        stock_missing.append(day)

# Hard assertion.
existing_in_missing = [
    day
    for day in stock_missing
    if parquet_path(STOCK_DIR, day).exists()
]

if existing_in_missing:

    raise RuntimeError(
        "ABORT: dates classified as missing already have Parquet files:\n"
        + "\n".join(
            str(parquet_path(STOCK_DIR, d))
            for d in existing_in_missing[:100]
        )
    )

assert (
    len(stock_existing)
    + len(stock_missing)
    == len(candidates)
)

print("=" * 90)
print("STOCK FILESYSTEM INVENTORY")
print("=" * 90)
print(f"Candidate dates   : {len(candidates):,}")
print(f"Existing Parquets : {len(stock_existing):,}")
print(f"Missing Parquets  : {len(stock_missing):,}")


In [ ]:
# 9. STOCK MANIFEST CLASSIFICATION
#
# Manifest is diagnostic only.
# It NEVER determines whether an existing Parquet is downloaded.
#
# Any missing Parquet is eligible for download/retry.

latest_stock_manifest = load_latest_manifest(
    STOCK_MANIFEST
)

manifest_downloaded_but_missing = [
    day
    for day in stock_missing
    if latest_stock_manifest.get(
        iso(day),
        {},
    ).get("status") == "downloaded"
]

manifest_failed = [
    day
    for day in stock_missing
    if latest_stock_manifest.get(
        iso(day),
        {},
    ).get("status") in {
        "failed",
        "error",
    }
]

manifest_not_found = [
    day
    for day in stock_missing
    if latest_stock_manifest.get(
        iso(day),
        {},
    ).get("status") == "not_found"
]

# THIS is the actual queue.
stock_download_queue = list(
    stock_missing
)

# Hard assertion.
assert len(stock_download_queue) == len(stock_missing)

assert all(
    not parquet_path(STOCK_DIR, day).exists()
    for day in stock_download_queue
)

print("=" * 90)
print("STOCK DOWNLOAD QUEUE")
print("=" * 90)
print(f"Queue size                         : {len(stock_download_queue):,}")
print(f"Previously downloaded but missing  : {len(manifest_downloaded_but_missing):,}")
print(f"Previously failed/error            : {len(manifest_failed):,}")
print(f"Previously not_found               : {len(manifest_not_found):,}")


In [ ]:
# 10. STOCK FINAL PREFLIGHT
#
# Absolutely NO network request is allowed if the queue contains
# an existing Parquet.

existing_in_queue = [
    day
    for day in stock_download_queue
    if parquet_path(STOCK_DIR, day).exists()
]

if existing_in_queue:

    raise RuntimeError(
        "ABORTED BEFORE ANY STOCK DOWNLOAD. "
        "Existing files found in queue:\n"
        + "\n".join(
            str(parquet_path(STOCK_DIR, d))
            for d in existing_in_queue[:100]
        )
    )

assert all(
    not parquet_path(STOCK_DIR, day).exists()
    for day in stock_download_queue
)

print("=" * 90)
print("STOCK DOWNLOAD PREFLIGHT")
print("=" * 90)
print(f"All candidates : {len(candidates):,}")
print(f"Existing       : {len(stock_existing):,}")
print(f"Missing        : {len(stock_missing):,}")
print(f"To download    : {len(stock_download_queue):,}")
print()
print("✓ Existing Parquets will NOT be requested.")
print("✓ Only missing dates will be requested.")


In [ ]:
# 11. STOCK DOWNLOAD LOOP — MISSING / FAILED ONLY

run_started = datetime.now()
run_results = []

for day in tqdm(
    stock_download_queue,
    desc="NSE missing/failed files",
):

    # Final safety check immediately before the network request.
    path = parquet_path(
        STOCK_DIR,
        day,
    )

    if path.exists():

        raise RuntimeError(
            f"ABORT: Parquet appeared before network request: {path}"
        )

    result = download_stock_day(day)

    run_results.append(result)

    append_manifest(result)

    if result["status"] in {
        "downloaded",
        "error",
        "failed",
    }:

        if REQUEST_SLEEP_SECONDS:
            time.sleep(
                REQUEST_SLEEP_SECONDS
            )

run_finished = datetime.now()

run_df = pd.DataFrame(
    run_results
)

print()
print("=" * 90)
print("STOCK SYNC COMPLETED")
print("=" * 90)
print("Started :", run_started)
print("Finished:", run_finished)
print("Elapsed :", run_finished - run_started)
print("Existing/skipped :", len(stock_existing))
print("Downloaded/retried:", len(run_results))


In [ ]:
# 12. STOCK POST-DOWNLOAD SUMMARY

stock_downloaded = [
    r for r in run_results
    if r.get("status") == "downloaded"
]

stock_not_found = [
    r for r in run_results
    if r.get("status") == "not_found"
]

stock_failed = [
    r for r in run_results
    if r.get("status") == "failed"
]

print("=" * 90)
print("STOCK RESULT SUMMARY")
print("=" * 90)
print(f"Downloaded : {len(stock_downloaded):,}")
print(f"Not found  : {len(stock_not_found):,}")
print(f"Failed     : {len(stock_failed):,}")

if stock_failed:
    print("\nFAILED DATES:")
    for result in stock_failed[:100]:
        print(
            result["date"],
            result.get("message"),
        )


# NIFTY 50

The current endpoint is:

`POST https://www.niftyindices.com/BackPage/getHistoricaldatatabletoString`

Payload:

```json
{
  "cinfo": "{'name':'NIFTY 50','startDate':'DD-MM-YYYY','endDate':'DD-MM-YYYY','indexName':'NIFTY 50'}"
}
```

The endpoint has been verified to return valid JSON, but it can return dates outside the requested range. The downloader therefore filters locally and refuses to save anything outside the requested range.


In [ ]:
# 13. NIFTY session

NIFTY_ENDPOINT = (
    "https://www.niftyindices.com/BackPage/"
    "getHistoricaldatatabletoString"
)

NIFTY_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "application/json, text/javascript, "
        "*/*; q=0.01"
    ),
    "Content-Type": (
        "application/json; charset=utf-8"
    ),
    "Origin": "https://www.niftyindices.com",
    "Referer": "https://www.niftyindices.com/reports",
}

nifty_session = requests.Session()
nifty_session.headers.update(
    NIFTY_HEADERS
)

try:
    response = nifty_session.get(
        "https://www.niftyindices.com/reports",
        timeout=30,
    )
    print(
        "NIFTY reports page:",
        response.status_code,
    )
except Exception as exc:
    print(
        "NIFTY session warning:",
        repr(exc),
    )


In [ ]:
# 14. NIFTY fetch + range diagnostic + hard filter

def fetch_nifty_range(
    session,
    requested_start,
    requested_end,
    sample_rows=20,
):

    requested_start = (
        pd.Timestamp(requested_start)
        .normalize()
    )

    requested_end = (
        pd.Timestamp(requested_end)
        .normalize()
    )

    start_str = requested_start.strftime(
        "%d-%m-%Y"
    )

    end_str = requested_end.strftime(
        "%d-%m-%Y"
    )

    cinfo = (
        "{'name':'NIFTY 50',"
        f"'startDate':'{start_str}',"
        f"'endDate':'{end_str}',"
        "'indexName':'NIFTY 50'}"
    )

    payload = {
        "cinfo": cinfo
    }

    print("=" * 90)
    print("NIFTY ENDPOINT DIAGNOSTIC")
    print("=" * 90)
    print(
        f"Requested range : "
        f"{requested_start.date()} -> "
        f"{requested_end.date()}"
    )
    print(
        f"Endpoint        : {NIFTY_ENDPOINT}"
    )

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            response = session.post(
                NIFTY_ENDPOINT,
                headers=NIFTY_HEADERS,
                json=payload,
                timeout=60,
            )

            print(
                "HTTP status     :",
                response.status_code,
            )
            print(
                "Content-Type    :",
                response.headers.get(
                    "Content-Type"
                ),
            )
            print(
                "Response size   :",
                f"{len(response.content):,}",
                "bytes",
            )

            response.raise_for_status()

            try:
                records = response.json()
            except Exception as exc:
                raise RuntimeError(
                    "NIFTY endpoint did not return JSON. "
                    f"First 500 chars: "
                    f"{response.text[:500]!r}"
                ) from exc

            if not isinstance(records, list):
                raise RuntimeError(
                    f"Unexpected NIFTY response type: "
                    f"{type(records)}"
                )

            print(
                "Records returned:",
                f"{len(records):,}",
            )

            if not records:
                return pd.DataFrame(
                    columns=STOCK_COLUMNS
                )

            df = pd.DataFrame(records)

            required = {
                "HistoricalDate",
                "OPEN",
                "HIGH",
                "LOW",
                "CLOSE",
            }

            missing = (
                required
                - set(df.columns)
            )

            if missing:
                raise RuntimeError(
                    "Missing NIFTY columns: "
                    f"{sorted(missing)}"
                )

            df["date"] = pd.to_datetime(
                df["HistoricalDate"],
                format="%d %b %Y",
                errors="coerce",
            ).dt.normalize()

            valid = df["date"].notna()

            if valid.any():

                actual_min = (
                    df.loc[
                        valid,
                        "date",
                    ].min()
                )

                actual_max = (
                    df.loc[
                        valid,
                        "date",
                    ].max()
                )

                print()
                print(
                    "Returned date range:"
                )
                print(
                    "  Earliest returned:",
                    actual_min.date(),
                )
                print(
                    "  Latest returned  :",
                    actual_max.date(),
                )

                before = df[
                    valid
                    & (
                        df["date"]
                        < requested_start
                    )
                ].copy()

                after = df[
                    valid
                    & (
                        df["date"]
                        > requested_end
                    )
                ].copy()

                out = pd.concat(
                    [before, after],
                    ignore_index=True,
                )

                print()
                print("-" * 90)
                print(
                    "OUT-OF-RANGE RECORD DIAGNOSTIC"
                )
                print("-" * 90)

                if out.empty:

                    print(
                        "✓ No out-of-range "
                        "records returned."
                    )

                else:

                    print(
                        f"WARNING: {len(out):,} "
                        "out-of-range records returned."
                    )

                    print(
                        "  Before requested start:",
                        len(before),
                    )

                    print(
                        "  After requested end   :",
                        len(after),
                    )

                    display_columns = [
                        "HistoricalDate",
                        "date",
                        "OPEN",
                        "HIGH",
                        "LOW",
                        "CLOSE",
                    ]

                    display_columns = [
                        c
                        for c in display_columns
                        if c in out.columns
                    ]

                    print(
                        "\nSample out-of-range records:"
                    )

                    print(
                        out
                        .sort_values("date")
                        [
                            display_columns
                        ]
                        .head(sample_rows)
                        .to_string(
                            index=False
                        )
                    )

                    if not before.empty:
                        print(
                            "Earliest out-of-range:",
                            before["date"]
                            .min()
                            .date(),
                        )

                    if not after.empty:
                        print(
                            "Latest out-of-range:",
                            after["date"]
                            .max()
                            .date(),
                        )

            # ------------------------------------------------
            # STRICT LOCAL DATE FILTER
            # ------------------------------------------------

            in_range = (
                valid
                & (
                    df["date"]
                    >= requested_start
                )
                & (
                    df["date"]
                    <= requested_end
                )
            )

            filtered = df.loc[
                in_range
            ].copy()

            print()
            print("-" * 90)
            print("FILTER RESULT")
            print("-" * 90)
            print(
                "Returned records :",
                len(df),
            )
            print(
                "In-range records :",
                len(filtered),
            )
            print(
                "Removed records  :",
                len(df) - len(filtered),
            )

            if filtered.empty:
                return pd.DataFrame(
                    columns=STOCK_COLUMNS
                )

            filtered["symbol"] = "NIFTY50"

            filtered["open"] = pd.to_numeric(
                filtered["OPEN"],
                errors="coerce",
            )

            filtered["high"] = pd.to_numeric(
                filtered["HIGH"],
                errors="coerce",
            )

            filtered["low"] = pd.to_numeric(
                filtered["LOW"],
                errors="coerce",
            )

            filtered["close"] = pd.to_numeric(
                filtered["CLOSE"],
                errors="coerce",
            )

            filtered["volume"] = pd.NA

            filtered = (
                filtered[
                    STOCK_COLUMNS
                ]
                .sort_values("date")
                .reset_index(drop=True)
            )

            # ------------------------------------------------
            # HARD RANGE ASSERTION
            # ------------------------------------------------

            bad = filtered[
                (
                    filtered["date"]
                    < requested_start
                )
                |
                (
                    filtered["date"]
                    > requested_end
                )
            ]

            if not bad.empty:
                raise RuntimeError(
                    "CRITICAL: out-of-range "
                    "NIFTY records survived filtering."
                )

            assert (
                filtered["date"].min()
                >= requested_start
            )

            assert (
                filtered["date"].max()
                <= requested_end
            )

            print(
                f"✓ Date-range validation passed: "
                f"{len(filtered):,} records strictly "
                f"within {requested_start.date()} -> "
                f"{requested_end.date()}"
            )

            return filtered

        except Exception as exc:

            last_error = repr(exc)

            if attempt < MAX_RETRIES:

                wait = min(
                    30,
                    2 ** (attempt - 1)
                    + random.random(),
                )

                print(
                    f"Attempt {attempt} failed; "
                    f"retrying in {wait:.1f}s"
                )

                time.sleep(wait)

    raise RuntimeError(
        f"NIFTY request failed after "
        f"{MAX_RETRIES} attempts: "
        f"{last_error}"
    )


In [ ]:
# 15. NIFTY FILESYSTEM INVENTORY
#
# Exactly the same filesystem-first rule as stocks.

nifty_candidates = list(
    date_range_days(
        START_DATE,
        END_DATE,
    )
)

nifty_existing = []
nifty_missing = []

for day in nifty_candidates:

    path = parquet_path(
        NIFTY_DIR,
        day,
    )

    if path.exists():
        nifty_existing.append(day)
    else:
        nifty_missing.append(day)

# Hard assertion.
existing_in_missing = [
    day
    for day in nifty_missing
    if parquet_path(
        NIFTY_DIR,
        day,
    ).exists()
]

if existing_in_missing:

    raise RuntimeError(
        "ABORT: NIFTY dates classified as missing "
        "already have Parquet files:\n"
        + "\n".join(
            str(
                parquet_path(
                    NIFTY_DIR,
                    d,
                )
            )
            for d in existing_in_missing[:100]
        )
    )

assert (
    len(nifty_existing)
    + len(nifty_missing)
    == len(nifty_candidates)
)

# Actual NIFTY download queue.
nifty_download_queue = list(
    nifty_missing
)

assert (
    len(nifty_download_queue)
    == len(nifty_missing)
)

assert all(
    not parquet_path(
        NIFTY_DIR,
        day,
    ).exists()
    for day in nifty_download_queue
)

print("=" * 90)
print("NIFTY 50 FILESYSTEM INVENTORY")
print("=" * 90)
print(
    f"Candidate dates   : "
    f"{len(nifty_candidates):,}"
)
print(
    f"Existing Parquets : "
    f"{len(nifty_existing):,}"
)
print(
    f"Missing Parquets  : "
    f"{len(nifty_missing):,}"
)
print(
    f"Download queue    : "
    f"{len(nifty_download_queue):,}"
)


In [ ]:
# 16. NIFTY FINAL PREFLIGHT
#
# If every NIFTY Parquet exists, this cell guarantees that
# the downloader will make ZERO NIFTY API requests.

nifty_existing_in_queue = [
    day
    for day in nifty_download_queue
    if parquet_path(
        NIFTY_DIR,
        day,
    ).exists()
]

if nifty_existing_in_queue:

    raise RuntimeError(
        "ABORTED BEFORE ANY NIFTY DOWNLOAD. "
        "Existing files found in queue:\n"
        + "\n".join(
            str(
                parquet_path(
                    NIFTY_DIR,
                    d,
                )
            )
            for d in nifty_existing_in_queue[:100]
        )
    )

assert all(
    not parquet_path(
        NIFTY_DIR,
        day,
    ).exists()
    for day in nifty_download_queue
)

print("=" * 90)
print("NIFTY 50 DOWNLOAD PREFLIGHT")
print("=" * 90)
print(
    f"All candidates : "
    f"{len(nifty_candidates):,}"
)
print(
    f"Existing       : "
    f"{len(nifty_existing):,}"
)
print(
    f"Missing        : "
    f"{len(nifty_missing):,}"
)
print(
    f"To download    : "
    f"{len(nifty_download_queue):,}"
)

if not nifty_download_queue:
    print()
    print(
        "✓ NOTHING TO DOWNLOAD"
    )
    print(
        "✓ ZERO NIFTY API requests "
        "will be made."
    )
else:
    print()
    print(
        "✓ Existing NIFTY files will NOT "
        "be requested."
    )
    print(
        "✓ Only missing NIFTY dates will "
        "be processed."
    )


In [ ]:
# 17. NIFTY CHUNK PLANNER
#
# IMPORTANT:
# Chunks are created ONLY over the missing-date range.
# A chunk with no missing dates is never requested.

def make_nifty_chunks(
    missing_dates,
    chunk_days=150,
):

    if not missing_dates:
        return []

    start = min(missing_dates)
    end = max(missing_dates)

    chunks = []
    cur = start

    while cur <= end:

        chunk_end = min(
            cur + timedelta(
                days=chunk_days - 1
            ),
            end,
        )

        chunks.append(
            (cur, chunk_end)
        )

        cur = (
            chunk_end
            + timedelta(days=1)
        )

    return chunks


nifty_chunks = make_nifty_chunks(
    nifty_download_queue,
    NIFTY_CHUNK_DAYS,
)

print(
    "NIFTY API chunks:",
    len(nifty_chunks),
)

if nifty_chunks:
    print(
        "First:",
        nifty_chunks[:3],
    )
    print(
        "Last :",
        nifty_chunks[-2:],
    )


In [ ]:
# 18. NIFTY DOWNLOAD LOOP — MISSING ONLY
#
# This is intentionally NOT:
#
#     for day in nifty_candidates
#
# and it is NOT:
#
#     for chunk in all historical chunks
#
# It processes only chunks that contain at least one
# missing Parquet date.

nifty_results = []

if (
    DOWNLOAD_NIFTY
    and nifty_download_queue
):

    missing_set = set(
        nifty_download_queue
    )

    for chunk_no, (
        chunk_start,
        chunk_end,
    ) in enumerate(
        nifty_chunks,
        1,
    ):

        # ----------------------------------------------------
        # Only dates that are still missing belong to this
        # request.
        # ----------------------------------------------------

        chunk_missing = [
            day
            for day in nifty_download_queue
            if (
                chunk_start
                <= day
                <= chunk_end
            )
        ]

        # NO missing files in this chunk:
        # do not call NIFTY API.
        if not chunk_missing:
            continue

        print()
        print("=" * 90)
        print(
            f"NIFTY CHUNK "
            f"{chunk_no}/{len(nifty_chunks)}"
        )
        print(
            f"API range     : "
            f"{chunk_start} -> {chunk_end}"
        )
        print(
            f"Missing dates : "
            f"{len(chunk_missing):,}"
        )
        print("=" * 90)

        # ----------------------------------------------------
        # Final race check before API request.
        # ----------------------------------------------------

        existing_now = [
            day
            for day in chunk_missing
            if parquet_path(
                NIFTY_DIR,
                day,
            ).exists()
        ]

        if existing_now:

            raise RuntimeError(
                "ABORT: NIFTY files appeared "
                "after queue creation:\n"
                + "\n".join(
                    str(
                        parquet_path(
                            NIFTY_DIR,
                            d,
                        )
                    )
                    for d in existing_now[:100]
                )
            )

        # ----------------------------------------------------
        # REQUEST
        # ----------------------------------------------------

        df = fetch_nifty_range(
            nifty_session,
            chunk_start,
            chunk_end,
        )

        if df.empty:

            print(
                "No in-range NIFTY records returned."
            )

            continue

        saved = 0

        # ----------------------------------------------------
        # SAVE ONLY MISSING DATES
        # ----------------------------------------------------

        for day, day_df in df.groupby(
            "date",
            sort=True,
        ):

            day = (
                pd.Timestamp(day)
                .date()
            )

            # The endpoint may return a date that exists
            # already. Never overwrite it.
            if day not in missing_set:
                continue

            path = parquet_path(
                NIFTY_DIR,
                day,
            )

            if path.exists():

                raise RuntimeError(
                    f"ABORT: NIFTY Parquet appeared "
                    f"before save: {path}"
                )

            if len(day_df) != 1:

                raise RuntimeError(
                    f"Expected exactly one NIFTY50 "
                    f"row for {day}; got "
                    f"{len(day_df)}"
                )

            day_df = day_df[
                STOCK_COLUMNS
            ].copy()

            atomic_to_parquet(
                day_df,
                path,
            )

            result = {
                "status": "downloaded",
                "date": iso(day),
                "rows": len(day_df),
                "source_start": iso(
                    chunk_start
                ),
                "source_end": iso(
                    chunk_end
                ),
            }

            nifty_results.append(
                result
            )

            append_jsonl(
                NIFTY_MANIFEST,
                result,
            )

            saved += 1

        print(
            f"Saved NIFTY files: "
            f"{saved:,}"
        )

        time.sleep(
            REQUEST_SLEEP_SECONDS
        )

else:

    print(
        "NIFTY download skipped — "
        "no missing Parquet files."
    )


In [ ]:
# 19. FINAL VALIDATION + SUMMARY

stock_downloaded = [
    r for r in run_results
    if r.get("status") == "downloaded"
]

stock_failed = [
    r for r in run_results
    if r.get("status") == "failed"
]

stock_not_found = [
    r for r in run_results
    if r.get("status") == "not_found"
]

nifty_downloaded = [
    r for r in nifty_results
    if r.get("status") == "downloaded"
]

# Final filesystem scan.
stock_files_final = sum(
    parquet_path(
        STOCK_DIR,
        d,
    ).exists()
    for d in candidates
)

stock_missing_final = [
    d
    for d in candidates
    if not parquet_path(
        STOCK_DIR,
        d,
    ).exists()
]

nifty_files_final = sum(
    parquet_path(
        NIFTY_DIR,
        d,
    ).exists()
    for d in nifty_candidates
)

nifty_missing_final = [
    d
    for d in nifty_candidates
    if not parquet_path(
        NIFTY_DIR,
        d,
    ).exists()
]

print("=" * 90)
print("FINAL SYNC SUMMARY")
print("=" * 90)

print()
print("STOCKS")
print(f"Candidate dates : {len(candidates):,}")
print(f"Files present   : {stock_files_final:,}")
print(f"Still missing   : {len(stock_missing_final):,}")
print(f"Downloaded now  : {len(stock_downloaded):,}")
print(f"Failed now      : {len(stock_failed):,}")
print(f"Not found now   : {len(stock_not_found):,}")

print()
print("NIFTY 50")
print(f"Candidate dates : {len(nifty_candidates):,}")
print(f"Files present   : {nifty_files_final:,}")
print(f"Still missing   : {len(nifty_missing_final):,}")
print(f"Downloaded now  : {len(nifty_downloaded):,}")

if stock_missing_final:
    print()
    print("STILL MISSING STOCK DATES — first 50:")
    print(
        [iso(d) for d in stock_missing_final[:50]]
    )

if nifty_missing_final:
    print()
    print("STILL MISSING NIFTY DATES — first 50:")
    print(
        [iso(d) for d in nifty_missing_final[:50]]
    )


In [ ]:
# 20. Validate all existing Parquets

stock_bad = []
nifty_bad = []

for day in candidates:

    path = parquet_path(
        STOCK_DIR,
        day,
    )

    if path.exists():

        ok, message = validate_parquet(
            path,
            STOCK_COLUMNS,
        )

        if not ok:
            stock_bad.append(
                (iso(day), message)
            )

for day in nifty_candidates:

    path = parquet_path(
        NIFTY_DIR,
        day,
    )

    if path.exists():

        ok, message = validate_parquet(
            path,
            STOCK_COLUMNS,
        )

        if not ok:
            nifty_bad.append(
                (iso(day), message)
            )

print("=" * 90)
print("PARQUET VALIDATION")
print("=" * 90)

print(
    "Bad stock Parquets:",
    len(stock_bad),
)

print(
    "Bad NIFTY Parquets:",
    len(nifty_bad),
)

if stock_bad:

    print()
    print("BAD STOCK FILES:")

    for item in stock_bad[:100]:
        print(item)

if nifty_bad:

    print()
    print("BAD NIFTY FILES:")

    for item in nifty_bad[:100]:
        print(item)


## Rerun behavior

The notebook is safe to rerun.

### Stock

If 4,100 dates are candidates and 3,900 Parquets exist:

- 3,900 are skipped
- only 200 are queued
- only those 200 can generate network requests

### NIFTY

If 4,100 dates are candidates and 4,050 Parquets exist:

- 4,050 are skipped
- only 50 are queued
- only API chunks containing those 50 dates are requested
- returned rows are filtered to the exact chunk range
- only the 50 missing dates can be written

If every NIFTY Parquet already exists:

**zero NIFTY API requests are made.**

The manifest is retained for diagnostics/retry visibility, but the filesystem remains the source of truth.
